# MCP Hosts, Clients, and Servers

MCP defines three actors with strict boundaries:

- **Server** — exposes tools, resources, and prompts (owned by a service team or vendor).
- **Client** — maintains one protocol session to exactly one server.
- **Host** — the AI application that embeds clients, runs the agent loop, and presents results to the user.

Confusing these roles is a common source of bugs: putting business logic in the host that belongs on the server, or sharing one client across multiple servers. This notebook explains each actor's responsibilities and implements a **ShopCo Support Host** with three server connections, consent gates, tool routing, and failure isolation — all in self-contained plain Python.

In [ ]:
"""
ShopCo MCP Host Lab — hosts, clients, and servers.
"""

import json
import re
import uuid
from dataclasses import dataclass, field
from datetime import datetime, timezone
from enum import Enum
from typing import Any, Callable


def pretty(obj: Any) -> str:
    return json.dumps(obj, indent=2, default=str)


ORDERS = {
    "ORD-1001": {"customer": "alice@shop.com", "status": "shipped", "amount": 1299.00},
    "ORD-1002": {"customer": "bob@shop.com", "status": "processing", "amount": 449.00},
}

POLICIES = [
    {"id": "ret-02", "text": "Returns within 30 days if unused."},
    {"id": "cancel-03", "text": "Cancellations allowed only before shipment."},
]

EVENT_LOG: list[dict[str, Any]] = []

print("ShopCo MCP Host Lab ready")

---

## 1. The Three Actors — Who Does What

```
┌─────────────────────────────────────────────────────────────────┐
│  HOST (ShopCo Support App)                                      │
│  • Runs agent loop                                              │
│  • User consent for server connections                          │
│  • Aggregates tools from all clients                            │
│  • Routes tool calls to correct client                          │
│  ┌──────────────┐  ┌──────────────┐  ┌──────────────┐          │
│  │   CLIENT 1   │  │   CLIENT 2   │  │   CLIENT 3   │          │
│  │  (session)   │  │  (session)   │  │  (session)   │          │
│  └──────┬───────┘  └──────┬───────┘  └──────┬───────┘          │
└─────────┼─────────────────┼─────────────────┼─────────────────┘
          │                 │                 │
          ▼                 ▼                 ▼
    Orders Server    Policies Server    Billing Server
    (tools)          (tools+resources)  (tools)
```

| Actor | Owns | Must NOT do |
|-------|------|-------------|
| **Server** | Business logic, data access, capability schemas | Run the agent loop; connect to other servers |
| **Client** | Transport, initialize handshake, protocol state | Aggregate multiple servers; UI |
| **Host** | Agent loop, UX, routing, consent | Implement tool business logic directly |

---

## 2. The 1:1 Rule — One Client Per Server

Each MCP **client** maintains exactly **one session** to **one server**. A host with three servers embeds **three clients**.

| Pattern | Valid? |
|---------|--------|
| 1 host → 1 client → 1 server | Yes |
| 1 host → 3 clients → 3 servers | Yes |
| 1 client → 2 servers | **No** |
| 2 hosts → 1 shared client | **No** |

The host merges capabilities at the application layer — not by multiplexing one client.

---

## 3. JSON-RPC Foundation

Clients and servers communicate via JSON-RPC 2.0 messages.

In [ ]:
class RpcKind(str, Enum):
    REQUEST = "request"
    RESPONSE = "response"
    ERROR = "error"
    NOTIFICATION = "notification"


@dataclass
class RpcMessage:
    id: str | None
    method: str
    kind: RpcKind
    params: dict[str, Any] = field(default_factory=dict)
    result: Any = None
    error: dict[str, Any] | None = None

---

## 4. MCP Server — Capability Provider

A server is a **standalone process or service** that registers tools and handles protocol methods. It knows nothing about which host or agent is calling.

In [ ]:
@dataclass
class ServerTool:
    name: str
    description: str
    input_schema: dict[str, Any]
    handler: Callable[..., Any]


@dataclass
class ServerResource:
    uri: str
    name: str
    description: str
    reader: Callable[[], str]


class MCPServer:
    """MCP server — exposes capabilities, handles JSON-RPC."""

    PROTOCOL_VERSION = "2024-11-05"

    def __init__(self, name: str, version: str = "1.0.0"):
        self.name = name
        self.version = version
        self.tools: dict[str, ServerTool] = {}
        self.resources: dict[str, ServerResource] = {}
        self._sessions: set[str] = set()

    def add_tool(self, tool: ServerTool) -> None:
        self.tools[tool.name] = tool

    def add_resource(self, resource: ServerResource) -> None:
        self.resources[resource.uri] = resource

    def handle(self, msg: RpcMessage) -> RpcMessage | None:
        if msg.method == "initialize":
            session_id = msg.params.get("clientInfo", {}).get("name", "unknown")
            self._sessions.add(session_id)
            return RpcMessage(msg.id, msg.method, RpcKind.RESPONSE, result={
                "protocolVersion": self.PROTOCOL_VERSION,
                "capabilities": {
                    "tools": {"listChanged": bool(self.tools)},
                    "resources": {"subscribe": False} if self.resources else {},
                },
                "serverInfo": {"name": self.name, "version": self.version},
            })

        if msg.method == "notifications/initialized":
            return None

        if msg.method == "tools/list":
            return RpcMessage(msg.id, msg.method, RpcKind.RESPONSE, result={
                "tools": [
                    {"name": t.name, "description": t.description, "inputSchema": t.input_schema}
                    for t in self.tools.values()
                ]
            })

        if msg.method == "tools/call":
            name = msg.params.get("name", "")
            args = msg.params.get("arguments", {})
            tool = self.tools.get(name)
            if not tool:
                return RpcMessage(msg.id, msg.method, RpcKind.ERROR, error={"code": -32602, "message": f"Unknown tool: {name}"})
            result = tool.handler(**args)
            return RpcMessage(msg.id, msg.method, RpcKind.RESPONSE, result={
                "content": [{"type": "text", "text": json.dumps(result, default=str)}],
                "isError": False,
            })

        if msg.method == "resources/list":
            return RpcMessage(msg.id, msg.method, RpcKind.RESPONSE, result={
                "resources": [
                    {"uri": r.uri, "name": r.name, "description": r.description}
                    for r in self.resources.values()
                ]
            })

        if msg.method == "resources/read":
            uri = msg.params.get("uri", "")
            res = self.resources.get(uri)
            if not res:
                return RpcMessage(msg.id, msg.method, RpcKind.ERROR, error={"code": -32602, "message": f"Unknown resource: {uri}"})
            return RpcMessage(msg.id, msg.method, RpcKind.RESPONSE, result={
                "contents": [{"uri": uri, "mimeType": "application/json", "text": res.reader()}]
            })

        return RpcMessage(msg.id, msg.method, RpcKind.ERROR, error={"code": -32601, "message": f"Unknown method: {msg.method}"})

---

## 5. Build ShopCo Servers

Three independent servers — each could run as a separate subprocess in production.

In [ ]:
def build_orders_server() -> MCPServer:
    srv = MCPServer("shopco-orders", "1.0.0")
    srv.add_tool(ServerTool(
        "lookup_order", "Look up order by ORD-####",
        {"type": "object", "properties": {"order_id": {"type": "string"}}, "required": ["order_id"]},
        lambda order_id: {"found": order_id.upper() in ORDERS, "order_id": order_id.upper(), **ORDERS.get(order_id.upper(), {})},
    ))
    return srv


def build_policies_server() -> MCPServer:
    srv = MCPServer("shopco-policies", "1.0.0")
    srv.add_tool(ServerTool(
        "search_policy", "Search policies by keyword",
        {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]},
        lambda query: [p for p in POLICIES if any(t in p["text"].lower() for t in query.lower().split())] or POLICIES[:1],
    ))
    srv.add_resource(ServerResource(
        "shopco://policies/all", "all_policies", "Full policy list",
        lambda: pretty(POLICIES),
    ))
    return srv


def build_billing_server() -> MCPServer:
    srv = MCPServer("shopco-billing", "1.0.0")
    srv.add_tool(ServerTool(
        "lookup_balance", "Customer balance by email",
        {"type": "object", "properties": {"email": {"type": "string"}}, "required": ["email"]},
        lambda email: {"email": email, "balance": 12.50 if "bob" in email else 0.00},
    ))
    return srv


orders_srv = build_orders_server()
policies_srv = build_policies_server()
billing_srv = build_billing_server()

print("Servers:", orders_srv.name, policies_srv.name, billing_srv.name)

---

## 6. MCP Client — Session State Machine

A client tracks connection state and speaks JSON-RPC to exactly one server.

In [ ]:
class ClientState(str, Enum):
    DISCONNECTED = "disconnected"
    INITIALIZING = "initializing"
    READY = "ready"
    ERROR = "error"


class TransportKind(str, Enum):
    IN_PROCESS = "in_process"   # this notebook
    STDIO = "stdio"             # subprocess in production
    SSE = "sse"                 # remote HTTP


@dataclass
class ClientSessionInfo:
    server_name: str
    state: ClientState
    transport: TransportKind
    tools_count: int = 0
    resources_count: int = 0
    last_error: str = ""


class MCPClient:
    """One client → one server. Manages session lifecycle."""

    def __init__(
        self,
        server: MCPServer,
        client_name: str,
        transport: TransportKind = TransportKind.IN_PROCESS,
    ):
        self._server = server
        self._client_name = client_name
        self._transport = transport
        self._state = ClientState.DISCONNECTED
        self._tool_cache: list[dict] | None = None

    @property
    def session_info(self) -> ClientSessionInfo:
        return ClientSessionInfo(
            server_name=self._server.name,
            state=self._state,
            transport=self._transport,
            tools_count=len(self._tool_cache or []),
        )

    def connect(self) -> dict[str, Any]:
        self._state = ClientState.INITIALIZING
        try:
            resp = self._request("initialize", {
                "protocolVersion": MCPServer.PROTOCOL_VERSION,
                "capabilities": {},
                "clientInfo": {"name": self._client_name, "version": "1.0.0"},
            })
            self._notify("notifications/initialized", {})
            self._tool_cache = self._request("tools/list", {}).result.get("tools", [])
            self._state = ClientState.READY
            EVENT_LOG.append({"actor": "client", "event": "connected", "server": self._server.name})
            return resp.result
        except Exception as exc:
            self._state = ClientState.ERROR
            EVENT_LOG.append({"actor": "client", "event": "error", "server": self._server.name, "error": str(exc)})
            raise

    def disconnect(self) -> None:
        self._state = ClientState.DISCONNECTED
        self._tool_cache = None

    def _send(self, msg: RpcMessage) -> RpcMessage | None:
        if self._transport == TransportKind.IN_PROCESS:
            return self._server.handle(msg)
        raise NotImplementedError(f"Transport {self._transport} not simulated here")

    def _request(self, method: str, params: dict) -> RpcMessage:
        msg = RpcMessage(id=str(uuid.uuid4())[:8], method=method, kind=RpcKind.REQUEST, params=params)
        resp = self._send(msg)
        if resp is None:
            raise RuntimeError(f"No response for {method}")
        if resp.kind == RpcKind.ERROR:
            raise RuntimeError(resp.error.get("message", "error"))
        return resp

    def _notify(self, method: str, params: dict) -> None:
        self._send(RpcMessage(id=None, method=method, kind=RpcKind.NOTIFICATION, params=params))

    def list_tools(self) -> list[dict]:
        if self._state != ClientState.READY:
            raise RuntimeError(f"Client not ready: {self._state}")
        return self._tool_cache or []

    def call_tool(self, name: str, arguments: dict) -> Any:
        resp = self._request("tools/call", {"name": name, "arguments": arguments})
        text = resp.result["content"][0]["text"]
        return json.loads(text)

    def read_resource(self, uri: str) -> str:
        return self._request("resources/read", {"uri": uri}).result["contents"][0]["text"]


demo_client = MCPClient(orders_srv, "shopco-host/orders")
init = demo_client.connect()
print(f"Client state: {demo_client.session_info.state.value}")
print(f"Tools: {[t['name'] for t in demo_client.list_tools()]}")

---

## 7. MCP Host — Orchestrator of Clients

The host embeds clients, requests user consent, aggregates tools, and routes agent calls.

In [ ]:
@dataclass
class ServerRegistration:
    alias: str
    server: MCPServer
    description: str
    approved: bool = False


@dataclass
class QualifiedTool:
    server_alias: str
    tool_name: str
    qualified_name: str
    description: str
    input_schema: dict[str, Any]


class MCPHost:
    """ShopCo support application — embeds MCP clients."""

    def __init__(self, name: str = "shopco-support-host"):
        self.name = name
        self._registrations: dict[str, ServerRegistration] = {}
        self._clients: dict[str, MCPClient] = {}

    def register_server(self, alias: str, server: MCPServer, description: str) -> None:
        self._registrations[alias] = ServerRegistration(alias, server, description)

    def approve_server(self, alias: str) -> None:
        if alias not in self._registrations:
            raise KeyError(alias)
        self._registrations[alias].approved = True
        EVENT_LOG.append({"actor": "host", "event": "consent_granted", "server": alias})

    def connect(self, alias: str, transport: TransportKind = TransportKind.IN_PROCESS) -> ClientSessionInfo:
        reg = self._registrations.get(alias)
        if reg is None:
            raise KeyError(f"Unknown server alias: {alias}")
        if not reg.approved:
            raise PermissionError(f"User has not approved server: {alias}")
        client = MCPClient(reg.server, f"{self.name}/{alias}", transport)
        client.connect()
        self._clients[alias] = client
        EVENT_LOG.append({"actor": "host", "event": "client_connected", "alias": alias})
        return client.session_info

    def list_connected(self) -> list[ClientSessionInfo]:
        return [c.session_info for c in self._clients.values()]

    def aggregate_tools(self) -> list[QualifiedTool]:
        tools: list[QualifiedTool] = []
        for alias, client in self._clients.items():
            if client.session_info.state != ClientState.READY:
                continue
            for t in client.list_tools():
                tools.append(QualifiedTool(
                    server_alias=alias,
                    tool_name=t["name"],
                    qualified_name=f"{alias}__{t['name']}",
                    description=t["description"],
                    input_schema=t["inputSchema"],
                ))
        return tools

    def route_tool_call(self, qualified_name: str, arguments: dict) -> Any:
        if "__" not in qualified_name:
            raise ValueError(f"Expected qualified name alias__tool, got: {qualified_name}")
        alias, tool_name = qualified_name.split("__", 1)
        client = self._clients.get(alias)
        if client is None:
            raise KeyError(f"No connected client for alias: {alias}")
        return client.call_tool(tool_name, arguments)


host = MCPHost()
host.register_server("orders", orders_srv, "Order lookup API")
host.register_server("policies", policies_srv, "Policy search and docs")
host.register_server("billing", billing_srv, "Billing and balances")

print("Registered servers:", list(host._registrations.keys()))

---

## 8. User Consent — Host Responsibility

Before connecting, the host should obtain user approval (especially for servers with write access).

In [ ]:
def simulate_user_consent(host: MCPHost, aliases: list[str]) -> None:
    for alias in aliases:
        reg = host._registrations[alias]
        print(f"  [Consent UI] Connect to '{alias}'? — {reg.description}")
        host.approve_server(alias)


EVENT_LOG.clear()
simulate_user_consent(host, ["orders", "policies", "billing"])

for alias in ["orders", "policies", "billing"]:
    info = host.connect(alias)
    print(f"Connected {alias}: state={info.state.value} tools={info.tools_count}")

---

## 9. Host Tool Router — Agent Integration

The host converts aggregated MCP tools into schemas the agent loop understands.

In [ ]:
class HostToolRouter:
    """Routes agent tool calls through the host to the correct client."""

    def __init__(self, host: MCPHost):
        self._host = host
        self._tools = host.aggregate_tools()

    def openai_schemas(self) -> list[dict]:
        return [
            {
                "type": "function",
                "function": {
                    "name": t.qualified_name,
                    "description": f"[{t.server_alias}] {t.description}",
                    "parameters": t.input_schema,
                },
            }
            for t in self._tools
        ]

    def dispatch(self, qualified_name: str, arguments: dict) -> Any:
        return self._host.route_tool_call(qualified_name, arguments)


router = HostToolRouter(host)
print(f"Aggregated {len(router._tools)} tools from {len(host.list_connected())} clients:")
for t in router._tools:
    print(f"  {t.qualified_name}")

---

## 10. End-to-End — Agent Through Host

The agent never touches servers directly — only the host's router.

In [ ]:
class ShopCoSupportAgent:
    def __init__(self, router: HostToolRouter):
        self._router = router

    def run(self, query: str) -> dict[str, Any]:
        trace: list[str] = []
        parts: list[str] = []

        if "ord-" in query.lower():
            match = re.search(r"ORD-\d{4}", query.upper())
            oid = match.group(0) if match else "ORD-1001"
            data = self._router.dispatch("orders__lookup_order", {"order_id": oid})
            trace.append(f"host → orders client → lookup_order({oid})")
            if data.get("found"):
                parts.append(f"Order {oid} is {data.get('status')} (${data.get('amount', 0):.2f})")

        if any(w in query.lower() for w in ("return", "policy", "cancel")):
            hits = self._router.dispatch("policies__search_policy", {"query": query[:40]})
            trace.append("host → policies client → search_policy")
            parts.append(f"[{hits[0]['id']}] {hits[0]['text']}")

        if "bill" in query.lower() or "charge" in query.lower():
            bal = self._router.dispatch("billing__lookup_balance", {"email": "bob@shop.com"})
            trace.append("host → billing client → lookup_balance")
            parts.append(f"Balance for {bal['email']}: ${bal['balance']:.2f}")

        return {"answer": " | ".join(parts) or "How can I help?", "trace": trace}


agent = ShopCoSupportAgent(router)

for q in [
    "Where is ORD-1001?",
    "Can I return my order?",
    "Why was I charged on my bill?",
]:
    r = agent.run(q)
    print(f"\nQ: {q}")
    print(f"A: {r['answer']}")
    print(f"   Route: {r['trace']}")

---

## 11. Transport Modes — Where Clients Run

| Transport | Server location | Client behavior |
|-----------|-----------------|-----------------|
| **stdio** | Subprocess on same machine | Host spawns server; JSON-RPC over pipes |
| **SSE** | Remote URL | HTTP long-lived stream + POST |
| **In-process** | Same Python process | Direct method call (this notebook) |

The **client protocol logic is identical** — only the transport adapter changes.

In [ ]:
@dataclass
class TransportProfile:
    kind: TransportKind
    server_location: str
    latency_ms: float
    isolation: str


TRANSPORT_PROFILES = [
    TransportProfile(TransportKind.IN_PROCESS, "same process", 0.1, "none — dev only"),
    TransportProfile(TransportKind.STDIO, "subprocess", 2.0, "process boundary"),
    TransportProfile(TransportKind.SSE, "remote URL", 50.0, "network boundary"),
]

print(f"{'Transport':<12} {'Location':<14} {'Latency':<10} Isolation")
print("-" * 55)
for p in TRANSPORT_PROFILES:
    print(f"{p.kind.value:<12} {p.server_location:<14} {p.latency_ms:<10} {p.isolation}")

---

## 12. Failure Isolation — One Server Down

The host should handle a failed client without breaking other connections.

In [ ]:
class FailingServer(MCPServer):
    """Simulates a server that errors on tool calls."""
    def handle(self, msg: RpcMessage) -> RpcMessage | None:
        if msg.method == "tools/call":
            return RpcMessage(msg.id, msg.method, RpcKind.ERROR, error={"code": -32000, "message": "Server unavailable"})
        return super().handle(msg)


failing_srv = FailingServer("shopco-broken", "0.1.0")
failing_srv.add_tool(ServerTool("broken_tool", "Always fails", {"type": "object", "properties": {}}, lambda: {}))

host2 = MCPHost("resilience-demo")
host2.register_server("orders", orders_srv, "Orders")
host2.register_server("broken", failing_srv, "Broken server")
host2.approve_server("orders")
host2.approve_server("broken")
host2.connect("orders")
host2.connect("broken")

router2 = HostToolRouter(host2)

# Orders still works
ok = router2.dispatch("orders__lookup_order", {"order_id": "ORD-1001"})
print("Orders server:", "OK" if ok.get("found") else "FAIL")

# Broken server fails in isolation
try:
    router2.dispatch("broken__broken_tool", {})
except RuntimeError as e:
    print(f"Broken server (expected): {e}")

---

## 13. Real-World Mapping

| Production example | MCP role |
|--------------------|----------|
| Cursor IDE | **Host** |
| Cursor's MCP connection manager | **Client** (one per server) |
| `@modelcontextprotocol/server-filesystem` | **Server** |
| Claude Desktop | **Host** |
| Your FastAPI agent app | **Host** |
| ShopCo orders microservice exposing MCP | **Server** |

---

## 14. Responsibility Checklist

### Server author
- Define tool/resource schemas
- Implement handlers with validation
- Declare capabilities in `initialize`
- Never assume which host connects

### Client implementer (usually SDK)
- Manage initialize lifecycle
- Handle transport (stdio/SSE)
- Expose `list_tools`, `call_tool` to host
- One server per client instance

### Host application author
- User consent before connecting
- Aggregate tools with qualified names
- Route agent calls to correct client
- Run agent loop; handle server failures gracefully

---

## 15. Host Dashboard — Session Overview

In [ ]:
print(f"{'Alias':<12} {'Server':<18} {'State':<14} {'Transport':<12} Tools")
print("-" * 65)
for alias, client in host._clients.items():
    info = client.session_info
    print(f"{alias:<12} {info.server_name:<18} {info.state.value:<14} {info.transport.value:<12} {info.tools_count}")

print(f"\nEvent log ({len(EVENT_LOG)} entries):")
for e in EVENT_LOG[-8:]:
    print(f"  {e}")

---

## 16. Anti-Patterns

| Anti-pattern | Why it fails | Fix |
|--------------|--------------|-----|
| One client, many servers | Violates MCP session model | One client per server in host |
| Business logic in host | Can't swap servers | Move logic to server handlers |
| Skip user consent | Security risk | `approve_server()` gate |
| Unqualified tool names | Collisions across servers | `alias__tool_name` |
| Agent calls server directly | Bypasses routing and policy | Agent → host router → client |

---

## 17. Check Your Understanding

1. What is the **1:1 rule** for MCP clients and servers?
2. Name three responsibilities of the **host** vs the **server**.
3. What happens during client **initialize** vs **notifications/initialized**?
4. Why does the host use **qualified tool names**?
5. What is the difference between stdio and in-process transport?
6. How should the host handle one server failing?
7. Who owns user **consent** for connecting a new server?

---

## 18. Summary

MCP's three actors have clear boundaries:

- **Server** — exposes tools/resources; standalone capability provider.
- **Client** — one JSON-RPC session per server; manages transport and lifecycle.
- **Host** — embeds clients, obtains consent, aggregates tools, routes agent calls.

The ShopCo Support Host connected three servers through three clients, enforced consent, routed qualified tool calls, and isolated a failing server without breaking orders. Build servers for capabilities, clients for sessions, and hosts for the agent experience.